# MindScope — OpenML Student Depression Dataset

This notebook uses **OpenML dataset 46874 (version 2)**, a separate dataset from the 4,464-row Kaggle file. OpenML reports 27,901 rows and a nominal `Depression` target. Enable Internet in Kaggle to download it.

**Educational research only.** The model predicts the dataset's recorded label; it does not diagnose depression or recommend treatment. The source documentation does not fully establish how the label was constructed. See OpenML ID 46874 and the [dataset record](https://www.openml.org/search?id=46874&sort=runs&type=data).

## 1. Install/import and settings

In [ ]:
# Kaggle: Internet must be enabled for OpenML access.
%pip -q install openml catboost xgboost

import json, re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    brier_score_loss)
from sklearn.calibration import CalibrationDisplay
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATASET_ID = 46874  # OpenML version 2; version 1 is a different OpenML record.
TARGET_COLUMN = 'Depression'
FILTER_TO_STUDENTS = True
EXCLUDE_SENSITIVE_MENTAL_HEALTH_INPUTS = True

## 2. Automatically download from OpenML and inspect schema
No Kaggle dataset attachment is needed. This cell downloads OpenML dataset 46874 when Internet is enabled, then saves a CSV copy under `/kaggle/working/`.

In [ ]:
# Explicit download from OpenML. OpenML caches the ARFF locally in the notebook runtime.
print(f"Downloading OpenML dataset ID {DATASET_ID} ...", flush=True)
dataset = openml.datasets.get_dataset(
    DATASET_ID,
    download_data=True,
    download_qualities=False,
    download_features_meta_data=True
)
print("Download complete:", dataset.name, "| ID:", dataset.dataset_id)
print("Default target recorded by OpenML:", dataset.default_target_attribute)

X_raw, y_raw, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=TARGET_COLUMN
)
if y_raw is None:
    raise ValueError(
        f"OpenML did not return target {TARGET_COLUMN!r}. "
        "Inspect dataset.default_target_attribute and set TARGET_COLUMN accordingly."
    )
X_raw = X_raw.copy()
y_raw = pd.Series(y_raw, index=X_raw.index, name=TARGET_COLUMN)
df_download = X_raw.copy()
df_download[TARGET_COLUMN] = y_raw

csv_path = "/kaggle/working/student_depression_openml_46874.csv"
df_download.to_csv(csv_path, index=False)
print("Saved CSV copy to:", csv_path)
print("Predictor shape:", X_raw.shape)
print("Target distribution:")
display(y_raw.value_counts(dropna=False).rename("count").to_frame())
display(pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing": X_raw.isna().sum(),
    "unique": X_raw.nunique(dropna=False)
}))
display(X_raw.head())


## 3. Audit, population filter, and leakage/sensitivity exclusions

Check the record's feature meanings before interpreting results. The suicide-ideation question is excluded from this public demonstration because it is highly sensitive and may be a near-proxy for the target. We also remove identifiers and filter the profession field to students when that column exists.

In [ ]:
df = X_raw.copy()
df[TARGET_COLUMN] = y_raw
print('Exact duplicate rows:', int(df.duplicated().sum()))

# Keep student records only if the source has a profession column.
profession_col = next((c for c in df.columns if c.strip().lower() == 'profession'), None)
if FILTER_TO_STUDENTS and profession_col:
    vals = df[profession_col].astype(str).str.strip().str.casefold()
    print('Profession values:', sorted(df[profession_col].dropna().astype(str).unique())[:30])
    student_mask = vals.eq('student')
    if student_mask.any():
        df = df.loc[student_mask].copy()
        print('Filtered to student records:', len(df))
    else:
        print('No exact Student category found; leaving rows unchanged. Review profession values.')

# Remove IDs, target copies, and sensitive mental-health predictors.
def norm(s): return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()
exclude = []
for c in df.columns:
    n = norm(c)
    if c == TARGET_COLUMN:
        continue
    if n in {'id', 'student id', 'row id', 'instance'} or n.endswith(' id'):
        exclude.append(c)
    if EXCLUDE_SENSITIVE_MENTAL_HEALTH_INPUTS and any(term in n for term in [
        'suicidal thought', 'suicide', 'self harm', 'self harm history'
    ]):
        exclude.append(c)
exclude = sorted(set(exclude))
print('Excluded predictors:', exclude)

# Target audit; do not infer clinical truth from this source-defined label.
print('Missing target rows:', int(df[TARGET_COLUMN].isna().sum()))
df = df.dropna(subset=[TARGET_COLUMN]).drop_duplicates().reset_index(drop=True)
display(df[TARGET_COLUMN].value_counts(normalize=True).rename('proportion').to_frame())

# Save a compact dataset profile in Kaggle output.
profile = {
    'openml_dataset_id': DATASET_ID,
    'name': dataset.name,
    'rows_after_filtering': int(len(df)),
    'columns': list(df.columns),
    'excluded_predictors': exclude,
    'target': TARGET_COLUMN,
    'target_counts': {str(k): int(v) for k, v in df[TARGET_COLUMN].value_counts().items()},
    'missing_by_column': {str(k): int(v) for k, v in df.isna().sum().items()}
}
with open('/kaggle/working/openml_dataset_profile.json', 'w') as f:
    json.dump(profile, f, indent=2)
print('Saved /kaggle/working/openml_dataset_profile.json')

## 4. Split data and prepare categorical fields

The final test split is held out from model selection. Validation is used to choose the model and decision threshold. Preprocessing for XGBoost is fit inside its pipeline.

In [ ]:
X = df.drop(columns=[TARGET_COLUMN] + exclude, errors='ignore').copy()
y_text = df[TARGET_COLUMN].astype(str).str.strip()
le = LabelEncoder()
y = le.fit_transform(y_text)
print('Target mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
if len(le.classes_) != 2:
    raise ValueError(f'Expected a binary target; found {list(le.classes_)}')

# For consistent positive-class metrics, class 1 should be the positive label.
positive_candidates = [i for i, c in enumerate(le.classes_) if c.casefold() in {'yes', '1', 'true'}]
if positive_candidates and positive_candidates[0] != 1:
    y = 1 - y
    print('Re-encoded target so positive class is 1:', le.classes_[positive_candidates[0]])

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE,
    stratify=y_trainval
)

cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
for part in (X_train, X_val, X_test):
    for c in cat_cols:
        part[c] = part[c].fillna('__MISSING__').astype(str)
print('Split sizes:', len(X_train), 'train;', len(X_val), 'validation;', len(X_test), 'test')
print('Categorical:', cat_cols)
print('Numeric:', num_cols)

## 5. Compare a baseline, CatBoost, and XGBoost

CatBoost receives categorical columns natively. XGBoost uses fold-fitted imputation and one-hot encoding. This is a compact comparison rather than a large model search.

In [ ]:
pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
], remainder='drop')

models = {
    'Dummy': DummyClassifier(strategy='prior'),
    'CatBoost': CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=5,
        loss_function='Logloss', eval_metric='Logloss',
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
        thread_count=-1, cat_features=cat_cols
    ),
    'XGBoost': Pipeline([
        ('preprocess', pre),
        ('model', XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=2,
            eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

fitted = {}
rows = []
for name, model in models.items():
    print('Fitting', name, flush=True)
    if name == 'CatBoost':
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)
    fitted[name] = model
    prob = model.predict_proba(X_val)[:, 1]
    pred = (prob >= 0.5).astype(int)
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'precision_macro': precision_score(y_val, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_val, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_val, pred, average='macro'),
        'roc_auc': roc_auc_score(y_val, prob),
        'pr_auc': average_precision_score(y_val, prob),
        'brier': brier_score_loss(y_val, prob)
    })

results = pd.DataFrame(rows).sort_values('f1_macro', ascending=False)
display(results)
selected_name = results.iloc[0]['model']
selected_model = fitted[selected_name]
print('Selected using validation macro F1:', selected_name)

## 6. Select threshold on validation data only

In [ ]:
from sklearn.metrics import precision_recall_curve

val_prob = selected_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.10, 0.91, 0.01)
threshold_rows = []
for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'macro_f1': f1_score(y_val, pred, average='macro'),
        'yes_precision': precision_score(y_val, pred, zero_division=0),
        'yes_recall': recall_score(y_val, pred, zero_division=0)
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.sort_values('macro_f1', ascending=False).head(10))
selected_threshold = float(threshold_table.loc[threshold_table['macro_f1'].idxmax(), 'threshold'])
print('Validation-selected threshold:', round(selected_threshold, 2))

## 7. One final evaluation on the held-out test set

In [ ]:
test_prob = selected_model.predict_proba(X_test)[:, 1]
test_pred_05 = (test_prob >= 0.5).astype(int)
test_pred_selected = (test_prob >= selected_threshold).astype(int)

print('Selected model:', selected_name)
print('Threshold selected on validation:', round(selected_threshold, 2))
print('Test ROC-AUC:', round(roc_auc_score(y_test, test_prob), 4))
print('Test PR-AUC:', round(average_precision_score(y_test, test_prob), 4))
print('Test Brier score:', round(brier_score_loss(y_test, test_prob), 4))
print('\nTest report at default threshold 0.5:')
print(classification_report(y_test, test_pred_05, target_names=['No', 'Yes'], zero_division=0))
print('Test report at validation-selected threshold:')
print(classification_report(y_test, test_pred_selected, target_names=['No', 'Yes'], zero_division=0))

cm = confusion_matrix(y_test, test_pred_selected)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Greens')
plt.title(f'{selected_name} — test confusion matrix (threshold {selected_threshold:.2f})')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks([0, 1], ['No', 'Yes']); plt.yticks([0, 1], ['No', 'Yes'])
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha='center', va='center')
plt.colorbar(); plt.tight_layout(); plt.show()

## 8. Global feature importance and limitations

Feature importance describes model behavior, not causes. This dataset is a different source and target from the 4,464-row Bangladesh dataset. Compare them only as separate experiments unless you can establish compatible labels and sampling methods. Excluding the suicidal-thought item also means this model intentionally does not use that sensitive input. Strong metrics do not establish clinical validity. Do not use predictions to diagnose, triage, or recommend treatment.

In [ ]:
if selected_name == 'CatBoost':
    importance = pd.Series(
        selected_model.get_feature_importance(), index=X_train.columns
    ).sort_values(ascending=False)
else:
    fitted_xgb = selected_model.named_steps['model']
    feature_names = selected_model.named_steps['preprocess'].get_feature_names_out()
    importance = pd.Series(
        fitted_xgb.feature_importances_, index=feature_names
    ).sort_values(ascending=False).head(20)

display(importance.head(20).rename('importance').to_frame())
importance.head(20).sort_values().plot(kind='barh', figsize=(8, 6), title='Top model feature importances')
plt.tight_layout(); plt.show()